
**Step 1 of 3 — Data Acquisition & Cleaning**

Downloads the ULB Credit Card Fraud Detection dataset from Kaggle, validates the expected schema, cleans the data, and outputs cleaned.csv.

How to run: ingest.ipynb → enrich.ipynb → load_mongo.ipynb

In [ ]:
#loading the content
def load_from_content(raw_path: str) -> None:
    if os.path.exists(raw_path):
        logging.info(f"Found {raw_path}.")
        print(f"Found {raw_path} — ready to load.")
    else:
        logging.error(f"{raw_path} not found in /content.")
        raise FileNotFoundError(
            f"'{raw_path}' not found. Please upload creditcard.csv to the "
            "Colab Files panel (left sidebar) before continuing."
        )

load_from_content("/content/creditcard.csv")

In [ ]:
#Configuration
#Expected columns in the raw ULB dataset
REQUIRED_COLUMNS = (
    ["Time"] +
    [f"V{i}" for i in range(1, 29)] +
    ["Amount", "Class"]
)
RAW_PATH = "creditcard.csv"
OUTPUT_PATH = "cleaned.csv"

In [ ]:
#Download Dataset from Kaggle
def download_dataset(raw_path: str) -> None:
    """Download the ULB fraud dataset from Kaggle using the Kaggle CLI."""
    if os.path.exists(raw_path):
        logging.info(f"{raw_path} already exists, skipping download.")
        print(f"{raw_path} already exists — skipping download.")
        return
    try:
        logging.info("Downloading dataset from Kaggle...")
        os.system("kaggle datasets download -d mlg-ulb/creditcardfraud --unzip")
        logging.info("Download complete.")
        print("Download complete.")
    except Exception as e:
        logging.error(f"Download failed: {e}")
        raise

download_dataset(RAW_PATH)

In [ ]:
#loading the raw data
def load_raw(path: str) -> pd.DataFrame:
    logging.info(f"Loading raw data from {path}")
    try:
        df = pd.read_csv(path)
        logging.info(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
        return df
    except FileNotFoundError as e:
        logging.error(f"File not found: {e}")
        raise

df_raw = load_raw(RAW_PATH)
df_raw.head()

In [ ]:
#validating the schema, and checking if all columns are present
def validate_schema(df: pd.DataFrame, required_columns: list) -> None:
    logging.info("Validating schema...")
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        logging.error(f"Missing columns: {missing}")
        raise ValueError(f"Schema validation failed. Missing columns: {missing}")
    logging.info("Schema validation passed.")
    print(f"Schema valid. All {len(required_columns)} expected columns present.")
validate_schema(df_raw, REQUIRED_COLUMNS)

In [ ]:
#Cleaning all the data (dropping duplicate rows, validating amount)
def clean(df: pd.DataFrame) -> pd.DataFrame:
    logging.info(f"Starting cleaning. Rows before: {len(df)}")
    df = df.drop_duplicates()
    logging.info(f"After dropping duplicates: {len(df)} rows")
    df = df.dropna()
    logging.info(f"After dropping nulls: {len(df)} rows")
    # Validate Class values
    invalid_class = df[~df["Class"].isin([0, 1])]
    if not invalid_class.empty:
        logging.warning(f"Dropping {len(invalid_class)} rows with invalid Class values.")
        df = df[df["Class"].isin([0, 1])]
    df["Class"] = df["Class"].astype(int)
    # Validate Amount
    invalid_amount = df[df["Amount"] < 0]
    if not invalid_amount.empty:
        logging.warning(f"Dropping {len(invalid_amount)} rows with negative Amount.")
        df = df[df["Amount"] >= 0]

    logging.info(f"Cleaning complete. Rows after: {len(df)}")
    logging.info(f"Fraud rate: {df['Class'].mean():.4%}")
    return df
df_clean = clean(df_raw)
df_clean.head()

In [ ]:
#Summary Statitics
print(df_clean["Class"].value_counts())
print(df_clean["Amount"].describe().round(2))

In [ ]:
#saving the clean data
df_clean.to_csv(OUTPUT_PATH, index=False)
logging.info(f"Cleaned data saved to {OUTPUT_PATH}")